# Notebook 1: Baseline — PC and GES on Complete Data

This notebook establishes the baseline structural accuracy of PC and GES on the Asia and Sachs benchmark networks using complete (no missingness) synthetic data. These results are the reference point against which all missingness conditions are compared.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import config
from src.data.loader import load_network, sample_data, get_true_edges
from src.algorithms.pc_wrapper import run_pc
from src.algorithms.ges_wrapper import run_ges
from src.evaluation.metrics import evaluate
from src.evaluation.bootstrap import bootstrap_run, summarise_bootstrap

## 1. Load networks and sample data

In [ ]:
results = {}

for dataset_name in config.DATASETS:
    print(f"\n--- {dataset_name.upper()} ---")
    network = load_network(dataset_name)
    df = sample_data(network)
    true_edges = get_true_edges(network)
    all_nodes = list(network.nodes())

    print(f"Nodes: {all_nodes}")
    print(f"True edges: {true_edges}")
    print(f"Sample shape: {df.shape}")

    results[dataset_name] = {
        "network": network,
        "df": df,
        "true_edges": true_edges,
        "all_nodes": all_nodes,
    }

## 2. Single-run baseline — PC and GES

In [ ]:
for dataset_name, data in results.items():
    df = data["df"]
    true_edges = data["true_edges"]
    all_nodes = data["all_nodes"]

    pc_edges = run_pc(df)
    ges_edges = run_ges(df)

    pc_metrics = evaluate(true_edges, pc_edges, all_nodes)
    ges_metrics = evaluate(true_edges, ges_edges, all_nodes)

    print(f"\n=== {dataset_name.upper()} ===")
    print(f"PC  → SHD={pc_metrics['shd']}, FP={pc_metrics['fp_rate']:.2f}, FN={pc_metrics['fn_rate']:.2f}")
    print(f"GES → SHD={ges_metrics['shd']}, FP={ges_metrics['fp_rate']:.2f}, FN={ges_metrics['fn_rate']:.2f}")

    data["pc_edges"] = pc_edges
    data["ges_edges"] = ges_edges

## 3. Bootstrap baseline (30 iterations)

In [ ]:
bootstrap_results = {}

for dataset_name, data in results.items():
    df = data["df"]
    true_edges = data["true_edges"]
    all_nodes = data["all_nodes"]

    print(f"\nBootstrapping {dataset_name.upper()} ...")

    pc_boot = bootstrap_run(df, run_pc, true_edges, all_nodes)
    ges_boot = bootstrap_run(df, run_ges, true_edges, all_nodes)

    pc_summary = summarise_bootstrap(pc_boot)
    ges_summary = summarise_bootstrap(ges_boot)

    print(f"PC  baseline: SHD {pc_summary['shd_mean']:.2f} ± {pc_summary['shd_std']:.2f}")
    print(f"GES baseline: SHD {ges_summary['shd_mean']:.2f} ± {ges_summary['shd_std']:.2f}")

    bootstrap_results[dataset_name] = {
        "pc": {"raw": pc_boot, "summary": pc_summary},
        "ges": {"raw": ges_boot, "summary": ges_summary},
    }

## 4. Save baseline results

In [ ]:
import os
os.makedirs(f"../{config.RESULTS_DIR}", exist_ok=True)

for dataset_name, data in bootstrap_results.items():
    for algo in ["pc", "ges"]:
        path = f"../{config.RESULTS_DIR}/baseline_{dataset_name}_{algo}.csv"
        data[algo]["raw"].to_csv(path, index=False)
        print(f"Saved: {path}")